In [0]:
# All import statements
from pyspark.sql.functions import col, date_format, sum, avg, count, countDistinct, round, year, month, to_date, when, rank 
from pyspark.sql.window import Window
import plotly.express as px
import plotly.graph_objects as go
from delta.tables import DeltaTable


# Phase 1 - Setup & Data Loading 

## 1. Create a Databricks workspace.
- Workspace named SuperStore_Dataset is created.

## 2. Create a new notebook.
- Notebook named Super_Store_Dataset is created.

## 3. Upload the dataset into Databricks. 
- The dataset is uploaded in csv format by creating new catalog, schema and volume. 

## 4. Read the CSV file using PySpark.
- spark.read.csv is used.

## 5. Infer schema automatically.  
- inferSchema is set as True

In [0]:
# Read the CSV file from the volume with automatic schema inference
df = spark.read.csv(
    "/Volumes/task_6_superstore/quickstart_schema/sandbox/Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"'
)

display(df)

## 6. Display first 10 rows.  

In [0]:
display(df.limit(10))

## 7. Check total number of rows. 

In [0]:
# Count the total number of rows in the dataset
total_rows = df.count()
print(f"Total number of rows: {total_rows:,}")

## 8. Check total number of columns.  

In [0]:
# Count the total number of columns in the dataset
total_columns = len(df.columns)
print(f"Total number of columns: {total_columns}")

## 9. Print dataset schema.  

In [0]:
# Display the inferred schema
df.printSchema()

## 10. Identify column data types.  

In [0]:
# Display column names and their data types
print("Column Data Types:")
for col_name, col_type in df.dtypes:
    print(f"{col_name:<25} : {col_type}")

# Phase 2 - Data Cleaning 

## 11. Check for null values in all columns.  

In [0]:
# Count null values for each column
print("Null Value Count by Column:")
for column_name in df.columns:
    null_count = df.filter(col(column_name).isNull()).count()
    print(f"{column_name:<25} : {null_count} nulls")

## 12. Find duplicate records.  

In [0]:
# Count total and distinct records
total_records = df.count()
distinct_records = df.distinct().count()
duplicates = total_records - distinct_records

print(f"Total records: {total_records:,}")
print(f"Distinct records: {distinct_records:,}")
print(f"Duplicate records: {duplicates:,}")

In [0]:
# Check for duplicates excluding Row ID column
columns_without_row_id = [col for col in df.columns if col != "Row ID"]

total_records = df.count()
distinct_records = df.select(columns_without_row_id).distinct().count()
duplicates = total_records - distinct_records

print(f"Total records: {total_records:,}")
print(f"Distinct records (excluding Row ID): {distinct_records:,}")
print(f"Duplicate records: {duplicates:,}")

## 13. Remove duplicate records.   

In [0]:
# Remove duplicates based on all columns except Row ID
columns_without_row_id = [col for col in df.columns if col != "Row ID"]

# Remove duplicates
df_clean = df.dropDuplicates(subset=columns_without_row_id)

# Show results
original_count = df.count()
clean_count = df_clean.count()
removed = original_count - clean_count

df = df_clean

print(f"Original records: {original_count:,}")
print(f"After removing duplicates: {clean_count:,}")
print(f"Duplicates removed: {removed:,}")

## 14. Convert Order Date to proper date format.  

In [0]:
# Convert Order Date to MM-dd-yyyy format
df = df.withColumn("Order Date", date_format(col("Order Date"), "MM-dd-yyyy"))

display(df.limit(10))

## 15. Convert Ship Date to proper date format.  

In [0]:
# Convert Ship Date to MM-dd-yyyy format
df = df.withColumn("Ship Date", date_format(col("Ship Date"), "MM-dd-yyyy"))

display(df.limit(10))

## 16. Check for invalid sales values.  

In [0]:
# Check for invalid sales values
negative_sales = df.filter(col("Sales") < 0).count()
zero_sales = df.filter(col("Sales") == 0).count()
null_sales = df.filter(col("Sales").isNull()).count()

print(f"Negative sales values: {negative_sales}")
print(f"Zero sales values: {zero_sales}")
print(f"Null sales values: {null_sales}")

## 17. Check for negative profit values.  

In [0]:
# Check for negative profit values
negative_profit = df.filter(col("Profit") < 0).count()
positive_profit = df.filter(col("Profit") > 0).count()
zero_profit = df.filter(col("Profit") == 0).count()

print(f"Negative profit (losses): {negative_profit}")
print(f"Positive profit: {positive_profit}")
print(f"Zero profit: {zero_profit}")

## 18. Create a cleaned DataFrame.  

In [0]:
# Create cleaned DataFrame
df_cleaned = df

print("Cleaned DataFrame Summary:")
print(f"Total records: {df_cleaned.count():,}")
print(f"Total columns: {len(df_cleaned.columns)}")
print("\nData cleaning completed successfully!")

## 19. Rename columns into standardized format.  

In [0]:
# Rename columns to snake_case standardized format
df_standardized = df

for old_name in df_standardized.columns:
    new_name = old_name.lower().replace(" ", "_").replace("-", "_")
    df_standardized = df_standardized.withColumnRenamed(old_name, new_name)

print("Standardized column names:")
for col_name in df_standardized.columns:
    print(f"  - {col_name}")

print(f"\nNew dataframe 'df_standardized' created with {df_standardized.count():,} records")
display(df_standardized.limit(10))

In [0]:
# Use standardized dataframe as the final dataframe
df = df_standardized

print(f"Final dataframe 'df' now has standardized column names")
print(f"Total records: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

## 20. Save cleaned data as a temporary view.  

In [0]:
# Create temporary view for SQL queries
df.createOrReplaceTempView("superstore_clean")

print(f"Temporary view 'superstore_clean' created successfully with {df.count()} records")

# Phase 3 - Basic Analysis 

## Storing the view as a dataframe and performing the operations as dataframe functions.

In [0]:
# Load the temporary view as a DataFrame
df_view = spark.table("superstore_clean")

## 21. Find total sales.  

In [0]:
# Find the total sales
total_sales = df_view.select(round(sum("sales"),2).alias("Total_Sales"))
display(total_sales)

## 22. Find total profit.  

In [0]:
# Find the total profit
total_profit = df_view.select(round(sum("profit"), 2).alias("Total_Profit"))
display(total_profit)

## 23. Find average sales per order.  

In [0]:
# Find the average sales per order
avg_sales_per_order = (
    df_view.groupBy("order_id")
    .agg(round(avg("sales"), 2).alias("Average_Sales_Per_Order"))
    .orderBy("order_id")
)
display(avg_sales_per_order)

## 24. Find total number of customers.  

In [0]:
# Find total number of customers
total_customers = df_view.select(countDistinct("customer_id").alias("Total_Customers"))
display(total_customers)

## 25. Find total number of products.  

In [0]:
# Find total number of products
total_products = df_view.select(countDistinct("product_id").alias("Total_Products"))
display(total_products)

## 26. Find unique categories.  

In [0]:
# Find the unique categories
unique_categories = df_view.select("category").distinct().orderBy("category")
display(unique_categories)

## 27. Find unique sub-categories.  

In [0]:
# Find the unique sub-categories
unique_subcategories = df_view.select("sub_category").distinct().orderBy("sub_category")
display(unique_subcategories)

## 28. Find total orders by region.  

In [0]:
# Find the total number of orders by region
total_orders_by_region = (
    df_view.groupBy("region")
    .agg(countDistinct("order_id").alias("total_orders"))
    .orderBy(col("total_orders").desc())
)
display(total_orders_by_region)

## 29. Find total sales by region.  

In [0]:
# Find the total sales by region
total_sales_by_region = (
    df_view.groupBy("region")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
)
display(total_sales_by_region)

## 30. Find total profit by region.  

In [0]:
# Find the total profit by region
total_profit_by_region = (
    df_view.groupBy("region")
    .agg(round(sum("profit"), 2).alias("total_profit"))
    .orderBy(col("total_profit").desc())
)
display(total_profit_by_region)

# Phase 4 - Intermediate Analysis 

## 31. Find top 10 products by sales.  

In [0]:
# Find the top 10 products by sales
top_10_products_by_sales = (
    df_view.groupBy("product_name")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
    .limit(10)
)
display(top_10_products_by_sales)

## 32. Find bottom 10 products by profit.  

In [0]:
# Find bottom 10 products by profit
bottom_10_products_by_profit = (
    df_view.groupBy("product_name")
    .agg(round(sum("profit"), 2).alias("total_profit"))
    .orderBy(col("total_profit").asc())
    .limit(10)
)
display(bottom_10_products_by_profit)

## 33. Find most profitable category.  

In [0]:
# Find the most profitable category
most_profitable_category = (
    df_view.groupBy("category")
    .agg(round(sum("profit"), 2).alias("total_profit"))
    .orderBy(col("total_profit").desc())
    .limit(1)
)
display(most_profitable_category)

## 34. Find least profitable category.  

In [0]:
# Find the least profitable category
least_profitable_category = (
    df_view.groupBy("category")
    .agg(round(sum("profit"), 2).alias("total_profit"))
    .orderBy(col("total_profit").asc())
    .limit(1)
)
display(least_profitable_category)

## 35. Find highest selling sub-category.  

In [0]:
# Find the highest selling subcategory
highest_selling_subcategory = (
    df_view.groupBy("sub_category")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
    .limit(1)
)
display(highest_selling_subcategory)

## 36. Find average discount by category.  

In [0]:
# Find the average discount by category
avg_discount_by_category = (
    df_view.groupBy("category")
    .agg(round(avg("discount"), 2).alias("average_discount"))
    .orderBy("category")
)
display(avg_discount_by_category)

## 37. Find monthly sales trend.  

In [0]:
# Find the monthly sales Trend
monthly_sales_trend = (
    df_view.withColumn("order_date_parsed", to_date(col("order_date"), "MM-dd-yyyy"))
    .groupBy(
        year("order_date_parsed").alias("year"),
        month("order_date_parsed").alias("month"),
    )
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy("year", "month")
)
display(monthly_sales_trend)

## 38. Find yearly sales trend.  

In [0]:
# Find the yearly sales trend
yearly_sales_trend = (
    df_view.withColumn("order_date_parsed", to_date(col("order_date"), "MM-dd-yyyy"))
    .groupBy(year("order_date_parsed").alias("year"))
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy("year")
)
display(yearly_sales_trend)

## 39. Find top 5 customers by sales.  

In [0]:
# Find the top 5 customers by sales
top_5_customers_by_sales = (
    df_view.groupBy("customer_name")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
    .limit(5)
)
display(top_5_customers_by_sales)

## 40. Find top 5 loss-making products.  

In [0]:
# Find the top 5 loss making products
top_5_loss_making_products = (
    df_view.groupBy("product_name")
    .agg(round(sum("profit"), 2).alias("total_profit"))
    .filter(col("total_profit") < 0)
    .orderBy(col("total_profit").asc())
    .limit(5)
)
display(top_5_loss_making_products)

# Phase 5 - Advanced PySpark Tasks 

## 41. Create a new column for Profit Margin.  

In [0]:
# New column for profit margin
df_with_profit_margin = df_view.withColumn(
    "profit_margin", 
    round(col("profit") / col("sales"), 2)
)
display(df_with_profit_margin.limit(10))

## 42. Categorize orders into High/Medium/Low sales.  

In [0]:
# Categorize sales
df_with_sales_category = df_view.withColumn(
    "sales_category",
    when(col("sales") >= 500, "High")
    .when((col("sales") >= 100) & (col("sales") < 500), "Medium")
    .otherwise("Low")
)
display(df_with_sales_category.limit(10))

## 43. Use Window Functions for ranking products.  

In [0]:
window_spec = Window.partitionBy("category").orderBy(col("total_sales").desc())

# Rank products by sales
product_ranking = (
    df_view.groupBy("category", "product_name")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .withColumn("rank", rank().over(window_spec))
    .orderBy("category", "rank")
)
display(product_ranking)

## 44. Find cumulative sales over time.  

In [0]:
window_cumulative = Window.orderBy("order_date_parsed").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate cumulative sales
cumulative_sales = (
    df_view.withColumn("order_date_parsed", to_date(col("order_date"), "MM-dd-yyyy"))
    .groupBy("order_date_parsed")
    .agg(round(sum("sales"), 2).alias("daily_sales"))
    .withColumn("cumulative_sales", round(sum("daily_sales").over(window_cumulative), 2))
    .orderBy("order_date_parsed")
)
display(cumulative_sales)

## 45. Find moving average of sales.  

In [0]:
window_moving_avg = Window.orderBy("order_date_parsed").rowsBetween(-6, 0)

# Calculate moving average of sales
moving_average_sales = (
    df_view.withColumn("order_date_parsed", to_date(col("order_date"), "MM-dd-yyyy"))
    .groupBy("order_date_parsed")
    .agg(round(sum("sales"), 2).alias("daily_sales"))
    .withColumn("moving_avg_7day", round(avg("daily_sales").over(window_moving_avg), 2))
    .orderBy("order_date_parsed")
)
display(moving_average_sales)

## 46. Partition data by region.  

In [0]:
# Partition data by region
partitioned_df = df_view.repartition("region")

print("Data partitioned by region")

# Show data distribution by region
region_distribution = partitioned_df.groupBy("region").count().orderBy("region")
display(region_distribution)

## 47. Cache frequently used DataFrames.  

In [0]:
print("Caching is not supported on serverless compute")

## 48. Explain the execution plan using .explain().  

In [0]:
complex_query = (
    df_view.groupBy("category", "region")
    .agg(
        round(sum("sales"), 2).alias("total_sales"),
        round(sum("profit"), 2).alias("total_profit")
    )
    .orderBy("category", col("total_sales").desc())
)

# Explain the query plan
complex_query.explain()

## 49. Compare performance before and after caching.  

In [0]:
print("Caching is not supported on serverless compute")

## 50. Write optimized PySpark queries.  

In [0]:
deltaTable = DeltaTable.forName(spark, "superstore_deltamain")

# Optimize and compact files
deltaTable.optimize().executeCompaction()
print("File compaction completed")

print("Table Details:")
details_df = spark.sql("DESCRIBE DETAIL superstore_deltamain")
display(details_df)

# Phase 6 - SQL Tasks in Databricks 

## 51. Create SQL tables from DataFrames.  

In [0]:
# Create SQL table
df_view.write.format("delta").mode("overwrite").saveAsTable("superstore_main")

In [0]:
# Create customer dimension table
df_customers = df_view.select("customer_id", "customer_name", "segment").distinct()
df_customers.write.format("delta").mode("overwrite").saveAsTable("dim_customers")

# Create product dimension table
df_products = df_view.select("product_id", "product_name", "category", "sub_category").distinct()
df_products.write.format("delta").mode("overwrite").saveAsTable("dim_products")

## 52. Run SQL queries for total sales.

In [0]:
%sql
-- total sales

SELECT 
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) as total_profit,
    COUNT(DISTINCT order_id) as total_orders
FROM superstore_main


## 53. Run SQL queries for top customers.  

In [0]:
%sql
-- top customers

SELECT 
    customer_name,
    ROUND(SUM(sales), 2) as total_sales,
    COUNT(DISTINCT order_id) as order_count
FROM superstore_main
GROUP BY customer_name
ORDER BY total_sales DESC
LIMIT 10

## 54. Run SQL queries for monthly trends.  

In [0]:
%sql
-- monthly trends

SELECT 
    YEAR(TO_DATE(order_date, 'MM-dd-yyyy')) as year,
    MONTH(TO_DATE(order_date, 'MM-dd-yyyy')) as month,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(AVG(sales), 2) as avg_sales
FROM superstore_main
GROUP BY year, month
ORDER BY year, month

## 55. Use GROUP BY and HAVING clauses.  

In [0]:
%sql
-- GROUP BY and HAVING clauses

SELECT 
    category,
    region,
    ROUND(SUM(sales), 2) as total_sales,
    COUNT(*) as order_count
FROM superstore_main
GROUP BY category, region
HAVING SUM(sales) > 50000
ORDER BY total_sales DESC

## 56. Use JOIN operations if multiple tables are created.  

In [0]:
%sql
-- JOIN operations if multiple tables are created
SELECT 
    c.customer_name,
    c.segment,
    p.category,
    ROUND(SUM(m.sales), 2) as total_sales
FROM superstore_main m
INNER JOIN dim_customers c ON m.customer_id = c.customer_id
INNER JOIN dim_products p ON m.product_id = p.product_id
GROUP BY c.customer_name, c.segment, p.category
ORDER BY total_sales DESC
LIMIT 20

## 57. Use CASE statements in SQL.  

In [0]:
%sql
-- CASE statements in SQL

SELECT 
    order_id,
    sales,
    profit,
    CASE 
        WHEN sales >= 500 THEN 'High'
        WHEN sales >= 100 THEN 'Medium'
        ELSE 'Low'
    END as sales_category,
    CASE 
        WHEN profit > 0 THEN 'Profitable'
        WHEN profit = 0 THEN 'Break-even'
        ELSE 'Loss'
    END as profit_status
FROM superstore_main
LIMIT 20

## 58. Create SQL views.  

In [0]:
%sql
-- Create SQL views
CREATE OR REPLACE VIEW vw_sales_summary AS
SELECT 
    region,
    category,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) as total_profit,
    COUNT(DISTINCT order_id) as order_count
FROM superstore_main
GROUP BY region, category;

CREATE OR REPLACE VIEW vw_customer_metrics AS
SELECT 
    customer_id,
    customer_name,
    segment,
    ROUND(SUM(sales), 2) as lifetime_value,
    COUNT(DISTINCT order_id) as order_count,
    ROUND(AVG(sales), 2) as avg_order_value
FROM superstore_main
GROUP BY customer_id, customer_name, segment;

SELECT * FROM vw_sales_summary ORDER BY total_sales DESC;
SELECT * FROM vw_customer_metrics ORDER BY lifetime_value DESC LIMIT 10;


# Phase 7 - Visualization 

## 59. Create a sales by region chart.  

In [0]:
# Group sales by region and aggregate total sales
sales_by_region = (
    df_view.groupBy("region")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
    .toPandas()
)

# Create a bar chart for sales by region
fig1 = px.bar(
    sales_by_region,
    x="region",
    y="total_sales",
    title="Sales by Region",
    labels={"total_sales": "Total Sales ($)", "region": "Region"},
    color="region",
)
fig1.show()

## 60. Create monthly sales trend chart.  

In [0]:
# Parse order_date and aggregate sales by year and month
monthly_sales = (
    df_view.withColumn("order_date_parsed", to_date(col("order_date"), "MM-dd-yyyy"))
    .groupBy(
        year("order_date_parsed").alias("year"),
        month("order_date_parsed").alias("month"),
    )
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy("year", "month")
    .toPandas()
)

# Create a year-month column for plotting
monthly_sales["year_month"] = (
    monthly_sales["year"].astype(str)
    + "-"
    + monthly_sales["month"].astype(str).str.zfill(2)
)

# Plot monthly sales trend as a line chart
fig2 = px.line(
    monthly_sales,
    x="year_month",
    y="total_sales",
    title="Monthly Sales Trend",
    labels={"total_sales": "Total Sales ($)", "year_month": "Year-Month"},
    markers=True,
)
fig2.update_xaxes(tickangle=45)
fig2.show()

## 61. Create category-wise sales chart.  

In [0]:
# Aggregate sales by category and sort descending
category_sales = (
    df_view.groupBy("category")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
    .toPandas()
)

# Create a pie chart for category-wise sales distribution
fig3_pie = px.pie(
    category_sales,
    values="total_sales",
    names="category",
    title="Category-wise Sales Distribution",
    color_discrete_sequence=["#FF6B6B", "#4ECDC4", "#45B7D1"],
)
fig3_pie.show()

## 62. Create profit vs sales visualization.  

In [0]:
# Aggregate sales and profit by product, select top 50 products
profit_vs_sales = (
    df_view.groupBy("product_name")
    .agg(
        round(sum("sales"), 2).alias("total_sales"),
        round(sum("profit"), 2).alias("total_profit"),
    )
    .orderBy(col("total_sales").desc())
    .limit(50)
    .toPandas()
)

# Create scatter plot for profit vs sales
fig4 = px.scatter(
    profit_vs_sales,
    x="total_sales",
    y="total_profit",
    title="Profit vs Sales (Top 50 Products)",
    labels={"total_sales": "Total Sales ($)", "total_profit": "Total Profit ($)"},
    hover_data=["product_name"],
    color="total_profit",
    color_continuous_scale="RdYlGn",
    size="total_sales",
)
# Add horizontal line for break-even
fig4.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Break-even")
fig4.show()

## 63. Create top customer visualization.  

In [0]:
# Aggregate sales by customer and select top 10 customers
top_customers = (
    df_view.groupBy("customer_name")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
    .limit(10)
    .toPandas()
)

# Create horizontal bar chart for top customers by sales
fig5 = px.bar(
    top_customers,
    x="total_sales",
    y="customer_name",
    title="Top 10 Customers by Sales",
    labels={"total_sales": "Total Sales ($)", "customer_name": "Customer Name"},
    orientation="h",
    color="total_sales",
    color_continuous_scale="Greens",
)
fig5.update_yaxes(categoryorder="total ascending")
fig5.show()

# Phase 8 - Delta Lake & Optimization 

## 64. Save cleaned data as Delta Table.  

In [0]:
# Create a Delta table from the cleaned DataFrame
df_view.write.format("delta").mode("overwrite").saveAsTable("superstore_deltamain")

## 65. Perform overwrite and append operations.

In [0]:
# Overwrite Operation - Replace all existing data
initial_count = spark.table("superstore_deltamain").count()
print(f"Initial row count: {initial_count}")

# Filter data for a specific region and overwrite
west_region_data = df_view.filter(col("region") == "West")
west_count = west_region_data.count()

# Overwrite the entire table with only West region data
west_region_data.write.format("delta").mode("overwrite").saveAsTable("superstore_deltamain")
after_overwrite_count = spark.table("superstore_deltamain").count()
print(f"After OVERWRITE: {after_overwrite_count} rows (West region only)")

# Restore full dataset
df_view.write.format("delta").mode("overwrite").saveAsTable("superstore_deltamain")
print(f"Full dataset restored: {df_view.count()} rows")

In [0]:
# Append Operation - Add new data without removing existing
current_count = spark.table("superstore_deltamain").count()
print(f"Current row count: {current_count}")

# Create sample new data to append
new_data = df_view.filter((col("region") == "East") & (col("category") == "Technology")).limit(100)

# Append new data to existing table
new_data.write.format("delta").mode("append").saveAsTable("superstore_deltamain")
after_append_count = spark.table("superstore_deltamain").count()

print(f"After APPEND: {after_append_count} rows")
print(f"Added: {after_append_count - current_count} new rows")